In [42]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LogisticRegression
from sklearn.metrics import mean_squared_error
from sklearn.datasets import make_classification 
from sklearn.metrics import accuracy_score 
from sklearn.tree import DecisionTreeClassifier 
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, StackingClassifier, BaggingClassifier, GradientBoostingClassifier

# Regularization

In [ ]:
"""
1. Model Linear (yang sudah kamu pelajari)
Ini yang paling “jelas” bentuknya:
- Lasso Regression → L1
- Ridge Regression → L2
- Elastic Net Regularization → kombinasi

2. Logistic Regression (Classification)
Dia juga punya regularization:
    from sklearn.linear_model import LogisticRegression
    model = LogisticRegression(penalty="l1", solver="liblinear")

📌 Intinya:
- Sama seperti linear regression, tapi untuk klasifikasi

3. Decision Tree 🌳
Tree tidak pakai L1/L2, tapi tetap ada regularization dengan cara:
- batasi kedalaman tree (max_depth)
- batasi jumlah leaf (max_leaf_nodes)
- minimum sample (min_samples_split)

4. Random Forest 🌲
- Banyak tree (bagging)
- Random subset fitur

👉 Ini sendiri sudah bentuk regularization:
- mengurangi variance
- tidak overfit seperti single tree

5. Gradient Boosting
Regularization-nya lewat:
- learning rate kecil
- jumlah tree dibatasi
- depth kecil

6. Support Vector Machine (SVM)
Sangat kuat di regularization:
    👉 pakai parameter:
        - C (regularization strength)
"""

In [ ]:
""" 
1. Apa itu “bobot” (weight)?
Dalam model seperti linear regression:
    y = w1x1 + w2x2 + w3x3 + b
👉 bobot (weight) = angka di depan fitur (w₁, w₂, w₃)
Artinya:
- Bobot menunjukkan seberapa besar pengaruh fitur ke prediksi

2. Apa itu “menambahkan penalti ke loss”?
Normalnya, model belajar dengan meminimalkan:
- Mean Squared Error (MSE)
    Loss = error prediksi

Tanpa Regularization
Model hanya fokus:
    "Yang penting error sekecil mungkin"
👉 Masalah:
- Model bisa “ngikutin data terlalu detail” (overfitting)
- Bobot bisa jadi sangat besar

Dengan L1 Regularization
Model sekarang mikir 2 hal:
- Error kecil
- Bobot jangan besar

📌 Ini yang disebut:
    menambahkan penalti ke loss

3. Kenapa Bobot Besar “Dihukum”?
Karena bobot besar biasanya berarti:
- model terlalu sensitif ke data
- mudah overfitting

📌 Jadi:
Regularization = “memaksa model tetap sederhana”
"""

### Regularization L1 (Lasso)

In [ ]:
""" 
L1 Regularization menambahkan penalti ke loss:
Artinya:
- Model dihukum kalau bobot besar
- Beberapa bobot bisa jadi 0

Efek utama:
- Feature selection otomatis
- Model jadi lebih sederhana (sparse)

Insight Penting dari L1
    Kalau kamu jalankan, biasanya akan terlihat:
        Beberapa koefisien bisa jadi:
            umur : 0.0

📌 Artinya:
fitur itu dianggap tidak penting oleh model


Pengaruh Hyperparameter α (alpha)
Coba ubah:
- Lasso(alpha=0.1)   # lebih lemah
- Lasso(alpha=10.0)  # lebih kuat

Efek:
- alpha kecil → model lebih bebas (risk overfitting)
- alpha besar → banyak coef jadi 0 (lebih simpel)

Kapan Pakai L1?
Gunakan L1 kalau:
- fitur banyak
- ada fitur tidak penting
- ingin feature selection otomatis
"""

In [16]:
np.random.seed(42)

df = pd.DataFrame({
    "luas": np.random.randint(50, 150, 50),
    "kamar": np.random.randint(1, 5, 50),
    "jarak": np.random.randint(1, 20, 50),
    "umur": np.random.randint(1, 30, 50)
})

df['harga'] = (
    5000 * df['luas'] +
    10000 * df['kamar'] - 
    3000 * df['jarak'] - 
    2000 * df['umur'] + 
    np.random.randn(50) * 10000
)

X = df.drop('harga', axis=1)
y = df['harga']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=1.0))
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('model', Lasso())])

In [17]:
y_pred = pipeline.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print('RMSE:', rmse)

RMSE: 8327.982929384025


In [18]:
# Koefisien dari masing-masing kolom

model = pipeline.named_steps['model']

for feature, coef in zip(X.columns, model.coef_):
    print(feature, ":", coef)

luas : 141131.13256073793
kamar : 10120.787061194296
jarak : -14507.189833072684
umur : -15186.953171574964


In [19]:
# Feature Selection 

selected_features = [
    feature for feature, coef in zip(X.columns, model.coef_) if coef != 0
]

print('Selected features:', selected_features)

Selected features: ['luas', 'kamar', 'jarak', 'umur']


### Regularization L2 (Ridge)

In [ ]:
""" 
L2 Regularization menambahkan penalti kuadrat ke loss:
- Bobot besar dihukum lebih keras (karena dikuadratkan)
- Tapi jarang jadi 0

L2 = “mengecilkan semua bobot, bukan membuang fitur”

Gunakan L2 kalau:
    - Semua fitur kemungkinan penting
    - ada korelasi antar fitur
    - ingin model stabil (tidak ekstrim)

Insight Penting dari L2
    Hasilnya biasanya:
        - Semua fitur tetap ada
        - Tidak ada yang jadi 0
        - Nilai coef lebih kecil dari LinearRegression

Insight Penting dari L2
    Hasilnya biasanya:
        - Semua fitur tetap ada
        - Tidak ada yang jadi 0
        - Nilai coef lebih kecil dari LinearRegression
"""

In [21]:
np.random.seed(42)

df = pd.DataFrame({
    "luas": np.random.randint(50, 150, 50),
    "kamar": np.random.randint(1, 5, 50),
    "jarak": np.random.randint(1, 20, 50),
    "umur": np.random.randint(1, 30, 50)
})

df['harga'] = (
    5000 * df['luas'] +
    10000 * df['kamar'] - 
    3000 * df['jarak'] - 
    2000 * df['umur'] + 
    np.random.randn(50) * 10000
)

X = df.drop('harga', axis=1)
y = df['harga']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('model', Ridge())])

In [22]:
y_pred = pipeline.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print('RMSE:', rmse)

RMSE: 10018.905171436185


In [23]:
# Liat koefisien dari masing-masing kolom
model = pipeline.named_steps['model']

for feature, coef in zip(X.columns, model.coef_):
    print(feature, ":", coef)

luas : 137397.20600996973
kamar : 8913.538540062342
jarak : -13834.705929757045
umur : -14610.324628096761


### Elastic Net (L1 + L2)

In [ ]:
""" 
1. Elastic Net Regularization adalah gabungan:
- L1 (Lasso)
- L2 (Ridge)

2. Intuisi Sederhana
Bayangin:
- L1 → buang fitur (jadi 0)
- L2 → kecilkan semua bobot

👉 Elastic Net:
- beberapa fitur dibuang, sisanya distabilkan

3. Parameter Penting
    ElasticNet(alpha=1.0, l1_ratio=0.5)
alpha → seberapa kuat regularization
l1_ratio:
- 1.0 → pure L1
- 0.0 → pure L2
- 0.5 → 

4. Kapan Pakai Elastic Net?
Gunakan kalau:
- fitur banyak
- fitur saling berkorelasi
- Lasso terlalu agresif (terlalu banyak nol)
- Ridge terlalu “lembut” (tidak ada seleksi)
"""

In [25]:
np.random.seed(42)

df = pd.DataFrame({
    "luas": np.random.randint(50, 150, 50),
    "kamar": np.random.randint(1, 5, 50),
    "jarak": np.random.randint(1, 20, 50),
    "umur": np.random.randint(1, 30, 50),
})

df["harga"] = (
    5000 * df["luas"] +
    10000 * df["kamar"] -
    3000 * df["jarak"] -
    2000 * df["umur"] +
    np.random.randn(50) * 10000
)

X = df.drop("harga", axis=1)
y = df["harga"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(alpha=1.0, l1_ratio=0.5))
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('model', ElasticNet())])

In [26]:
y_pred = pipeline.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RMSE:", rmse)

RMSE: 49331.05685942017


In [27]:
# Liat koefisien
model = pipeline.named_steps["model"]

for feature, coef in zip(X.columns, model.coef_):
    print(feature, ":", coef)

luas : 92108.9664884261
kamar : -1669.2973462060588
jarak : -7444.1104869572855
umur : -8374.90723288074


In [28]:
# Feature Selection 
selected_features = [
    f for f, c in zip(X.columns, model.coef_)
    if abs(c) > 1e-3
]

print("Selected:", selected_features)

Selected: ['luas', 'kamar', 'jarak', 'umur']


# Ensemble Methods

In [ ]:
""" 
1. Inti dari Pernyataan Itu
Kalimat itu maksudnya:
    Daripada mengandalkan 1 model saja, kita gabungkan banyak model (predictor) → hasilnya biasanya lebih baik

👉 Kumpulan model ini disebut:
- Ensemble Learning
- Modelnya disebut: ensemble method

2. Apa itu “ensemble” secara sederhana?
👉 Ensemble = kumpulan model
Contoh:
- 1 model → prediksi sendiri
- 10 model → prediksi bareng → digabung

📌 Analogi:
- seperti voting banyak orang, bukan 1 orang saja

3. Kenapa Bisa Lebih Baik?

Karena:
- tiap model punya error sendiri
- kalau digabung → error bisa saling “menutup”

👉 hasilnya:
- lebih stabil
- lebih akurat
"""

In [ ]:
""" 
1. Jawaban Inti
Gunakan Ensemble Learning ketika:
    👉 model tunggal belum cukup bagus atau tidak stabil

2. Kondisi-Kondisi Kapan Pakai Ensemble

✅ 1. Model Overfitting (variance tinggi)
Contoh:
- train score tinggi
- validation score jelek

👉 Gunakan:
- Random Forest

📌 Kenapa?
- averaging banyak model → mengurangi variance

✅ 2. Model Tidak Stabil
Misalnya:
- Decision Tree (hasil bisa berubah kalau data sedikit berubah)

👉 Ensemble bikin:
- hasil lebih konsisten

✅ 3. Akurasi Masih Kurang
Kalau:
- sudah coba model biasa
- performa stagnan

👉 Gunakan:
- Gradient Boosting

📌 Ini sering jadi “upgrade” dari model sederhana

✅ 4. Data Kompleks / Non-linear
Kalau:
- hubungan fitur tidak linear
- banyak interaksi antar fitur

👉 Ensemble (tree-based) biasanya lebih powerful

✅ 5. Kompetisi / Production (butuh performa maksimal)
👉 Hampir semua solusi top:
- pakai ensemble
"""

In [ ]:
""" 
Cek materi di Hands-on Machine Learning with Scikit-Learn, Keras & TensorFlow Chapter 7
"""

### Voting Classifier 

In [31]:
# Data
X, y = make_classification(n_samples=200, n_features=5, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model individiu
log_clf = LogisticRegression()
tree_clf = DecisionTreeClassifier()
knn_clf = KNeighborsClassifier()

# Ensemble (Voting)
voting_clf = VotingClassifier(
    estimators=[
        ('lr', log_clf),
        ('dt', tree_clf),
        ('knn', knn_clf)
    ], voting='hard'
)

# Training 
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression()),
                             ('dt', DecisionTreeClassifier()),
                             ('knn', KNeighborsClassifier())])

In [32]:
# Evaluasi 
for clf in (log_clf, tree_clf, knn_clf, voting_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

LogisticRegression 0.875
DecisionTreeClassifier 0.8
KNeighborsClassifier 0.8
VotingClassifier 0.825


### Random Forest 

In [34]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, random_state=42)

In [35]:
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.85


### Bagging (Bootstrap Aggregating)

In [ ]:
""" 
Contoh paling umum:
- Random Forest

🔸 Intuisi
- Banyak model (biasanya decision tree)
- Dilatih paralel
- Pakai subset data yang berbeda (bootstrap)
- Hasil → di-average / voting

📌 Tujuan:
- mengurangi variance (overfitting)
"""

In [40]:
X, y = make_classification(n_samples=200, n_features=5, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Bagging 
bagging_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    bootstrap=True,
    random_state=42
)

bagging_clf.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.8,
                  n_estimators=100, random_state=42)

In [41]:
y_pred = bagging_clf.predict(X_test)
print('Accuracy Bagging:', accuracy_score(y_test, y_pred))

Accuracy Bagging: 0.85


### Boosting

In [ ]:
""" 
👉 Contoh paling umum:
- Gradient Boosting

🔸 Intuisi
- Model dilatih berurutan
- Setiap model:
    👉 fokus memperbaiki error sebelumnya

📌 Tujuan:
- mengurangi bias (underfitting)
"""

In [43]:
# Boosting 
boosting_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3
)

boosting_clf.fit(X_train, y_train)

GradientBoostingClassifier()

In [44]:
y_pred = boosting_clf.predict(X_test)
print('Accuracy Boosting:', accuracy_score(y_test, y_pred))

Accuracy Boosting: 0.825


### Stacking

In [ ]:
""" 
1. Apa itu Stacking?
Stacking adalah:
- Menggabungkan beberapa model, lalu hasilnya dipelajari lagi oleh model lain

2. Intuisi Sederhana
Kalau voting:
- model A → prediksi
- model B → prediksi
- model C → voting → hasil akhir

Kalau stacking:
- model A → prediksi \
- model B → prediksi  → masuk ke model baru → hasil akhir
- model C → prediksi /

👉 Model terakhir disebut:
- meta-model

3. Analogi
- Voting → “ambil suara terbanyak”
- Stacking → “ada supervisor yang menilai semua prediksi”

📌 Jadi:
- Stacking lebih “pintar” dari voting
"""

In [37]:
X, y = make_classification(n_samples=200, n_features=5, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

estimators = [
    ('lr', LogisticRegression()),
    ('dt', DecisionTreeClassifier()),
    ('knn', KNeighborsClassifier())
]

# Stacking 
stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression()
)

stacking_clf.fit(X_train, y_train)

StackingClassifier(estimators=[('lr', LogisticRegression()),
                               ('dt', DecisionTreeClassifier()),
                               ('knn', KNeighborsClassifier())],
                   final_estimator=LogisticRegression())

In [38]:
y_pred = stacking_clf.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))

Accuracy: 0.825


# Early Stopping

In [ ]:
""" 
1. Konsep Early Stopping
Early Stopping adalah:
    menghentikan training lebih awal saat performa di validation mulai memburuk

2. Kenapa Perlu Early Stopping?
Saat training model, biasanya terjadi pola seperti ini:
- Train error → terus turun 📉
- Validation error → turun dulu, lalu naik 📈

👉 Ini tanda:
- model mulai overfitting

3. Intuisi Sederhana
Training terlalu lama → model “hafal data”
Early stopping → “stop sebelum terlalu hafal”

📌 Jadi:
- Early stopping = cara mencegah overfitting tanpa ubah model

4. Komponen Penting
1. Validation set
- Dipakai untuk monitor performa

2. Metric
Contoh:
- loss
- RMSE
- accuracy

3. Patience
- Berapa lama kita “nunggu” sebelum berhenti

Contoh:
    patience = 3

👉 Artinya:
- kalau 3 epoch berturut-turut tidak membaik → stop

🔸 n_estimators
Maksimum jumlah iterasi (tree)

🔸 n_iter_no_change
👉 ini adalah patience
Contoh:
    n_iter_no_change = 5

Artinya:
- kalau 5 iterasi tidak ada improvement → stop

🔸 validation_fraction
- bagian data training yang dipakai sebagai validation internal

🔸 model.n_estimators_
👉 ini penting banget:
- jumlah tree yang benar-benar dipakai
- biasanya < 1000 (karena stop lebih awal)
"""